# TinyGPT - a mini ChatGPT brain you can train on a laptop

Every GPT model - large or small - is essentially a **Next-Token Predictor**: given some text, guess what comes next, add it, and repeat.

To keep things simple, our "tokens" are single **characters** (real GPTs use word pieces). We'll train it on Shakespeare so it learns to write in his style.

How data flows through the TinyGPT pipeline:
```
[Input Text: "To be or not to "]
       ↓
 1. Tokenizer (turns each character into an ID number)
       ↓
 2. Embedding + Positional Encoding (gives each number a meaning AND a seat number)
       ↓
 3. Transformer Blocks (Self-Attention lets each character look back at the earlier ones for context)
       ↓
 4. Output Linear Layer & Softmax (gives the odds for every possible next character)
       ↓
[Predicted next character: "b"]  → add it on and repeat to write "be"
```

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# ==========================================
# 1. SETUP AND HYPERPARAMETERS (the "settings knobs")
# ==========================================
torch.manual_seed(42)    # Fix the dice rolls so every run gives the same result

# Tiny hyperparameters for consumer laptops
batch_size = 32          # How many text snippets the model studies at once
block_size = 64          # Context window - how many characters it can "look back" at
max_iters = 3000         # Total training steps
learning_rate = 1e-3     # How big a correction the model makes after each mistake
eval_interval = 300      # Print a progress report every 300 steps
eval_iters = 50          # How many batches to average when measuring the loss (a steadier score)

# Use the fastest chip available: NVIDIA GPU -> Apple Silicon GPU -> normal CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print("Using device:", device)

n_embd = 128             # Size of each character's "meaning vector" (GPT-2 uses 768)
n_head = 4               # Number of attention heads - like 4 readers each looking for different clues
n_layer = 4              # Number of stacked transformer blocks (more layers = deeper thinking)


Using device: mps


In [2]:
import os
import urllib.request

# ==========================================
# 2. LOAD THE TEXT AND BUILD THE TOKENIZER
# ==========================================
# We train on "Tiny Shakespeare" - about 1 MB of Shakespeare plays.
# Step A: Download it once and save it next to this notebook
file_path = 'tiny_shakespeare.txt'
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()
print("Total characters in the book:", len(text))

# Step B: Our "vocabulary" is every unique character (letters, spaces, punctuation)
chars = sorted(set(text))
vocab_size = len(chars)
print("Vocabulary size:", vocab_size)

# Step C: Tokenizer - a simple lookup table between characters and ID numbers
stoi = {ch: i for i, ch in enumerate(chars)}   # "string to integer"  e.g. 'a' -> 39
itos = {i: ch for i, ch in enumerate(chars)}   # "integer to string"  e.g. 39 -> 'a'
encode = lambda s: [stoi[c] for c in s]          # text -> list of numbers
decode = lambda l: ''.join(itos[i] for i in l)   # list of numbers -> text
print(encode("To be"), "->", decode(encode("To be")))

# Step D: Turn the whole book into numbers, then split 90% for studying, 10% for the exam
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]   # The model learns from this part
val_data = data[n:]     # Held back to check it isn't just memorising


Total characters in the book: 1115394
Vocabulary size: 65
[32, 53, 1, 40, 43] -> To be


In [3]:
# ==========================================
# 3. TRANSFORMER BUILDING BLOCKS
# ==========================================

# Step A: One attention head - each character "asks" the earlier characters which of them matter
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)   # "What do I contain?"
        self.query = nn.Linear(n_embd, head_size, bias=False)   # "What am I looking for?"
        self.value = nn.Linear(n_embd, head_size, bias=False)   # "What will I share if picked?"
        # Lower-triangle mask: a character may look BACK but never peek at future characters
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)
        # Match every query against every key -> "how relevant is each earlier character to me?"
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5     # (B, T, T), scaled to keep numbers calm
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # Hide the future
        wei = F.softmax(wei, dim=-1)                            # Turn scores into percentages
        v = self.value(x)   # (B, T, head_size)
        return wei @ v      # Blend the values using those percentages

# Step B: Several heads in parallel - each can focus on a different clue (spelling, rhythm, names...)
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)   # Mix the heads' findings back together

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # Glue each head's answer side by side
        return self.proj(out)

# Step C: Feed-forward - after gathering info, each character "thinks it over" on its own
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # Expand to a bigger space to think in
            nn.ReLU(),                       # Keep positive signals, drop negative ones
            nn.Linear(4 * n_embd, n_embd),   # Shrink back to the original size
        )

    def forward(self, x):
        return self.net(x)

# Step D: One Transformer block = "talk to others" (attention) + "think alone" (feed-forward)
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head              # 128 / 4 = 32 numbers per head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)           # LayerNorm keeps numbers in a steady range
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # "x + ..." is a residual (shortcut) connection: keep the old info, ADD the new insight
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [4]:
# ==========================================
# 4. THE FULL TinyGPT MODEL
# ==========================================
class TinyGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. Token embedding - turns each character ID into a meaning vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # 2. Position embedding - tells the model WHERE each character sits (1st, 2nd, 3rd...)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # 3. Stacked Transformer Blocks (Self-Attention + Feed Forward)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head=n_head) for _ in range(n_layer)])

        # 4. Final normalization layer
        self.ln_f = nn.LayerNorm(n_embd)
        # 5. Output layer - turns each vector into a score for every character in the vocabulary
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape   # B = number of snippets, T = characters per snippet

        # Look up meaning vectors and add position info
        tok_emb = self.token_embedding_table(idx)                                  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb   # Each character now knows WHAT it is and WHERE it is

        # Pass through the Transformer blocks
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)   # Raw scores ("logits") for what the next character should be

        loss = None
        if targets is not None:
            # Flatten into one long list so cross-entropy can grade every guess at once
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)   # How wrong were the guesses? (lower = better)

        return logits, loss

    @torch.no_grad()   # No learning while writing - saves memory and time
    def generate(self, idx, max_new_tokens):
        # idx is a (B, T) grid of character IDs - the text so far
        for _ in range(max_new_tokens):
            # Crop to the last block_size characters (the model can't see further back)
            idx_cond = idx[:, -block_size:]
            # Get predictions
            logits, _ = self(idx_cond)
            # Keep only the prediction for the very last position
            logits = logits[:, -1, :]            # (B, C)
            # Softmax turns raw scores into probabilities that add up to 1
            probs = F.softmax(logits, dim=-1)    # (B, C)
            # Pick the next character randomly, weighted by those probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # Stick it on the end and repeat
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        return idx


In [5]:
# ==========================================
# 5. TRAINING HELPERS
# ==========================================

# Step A: Grab random snippets of text to study
def get_batch(split):
    source = train_data if split == 'train' else val_data
    # Pick batch_size random starting points in the book
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])          # The input snippet
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])  # Same snippet shifted by 1 = the "answers"
    return x.to(device), y.to(device)

# Step B: Measure the loss fairly by averaging over several batches (one batch is too noisy)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()                        # Switch to "exam mode"
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()                       # Back to "study mode"
    return out

# Example: for input 'To b' the answer is 'o be' - every position predicts its next character
xb, yb = get_batch('train')
print("Input :", repr(decode(xb[0][:20].tolist())))
print("Target:", repr(decode(yb[0][:20].tolist())))


Input : 'Paris!\nWhat said my '
Target: 'aris!\nWhat said my m'


In [6]:
# ==========================================
# 6. TRAIN THE MODEL
# ==========================================
model = TinyGPT(vocab_size).to(device)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} million parameters")

# AdamW = the "coach" that nudges every weight in the right direction after each mistake
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Before training, loss is around 4.2-4.4 (pure random guessing among 65 characters = 4.17)
for step in range(max_iters):
    # Progress report every eval_interval steps (and at the very end)
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step:4d} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f}")

    xb, yb = get_batch('train')          # Step A: get a batch of snippets
    _, loss = model(xb, yb)              # Step B: make guesses and score them
    optimizer.zero_grad(set_to_none=True)  # Step C: wipe old corrections
    loss.backward()                      # Step D: work out which weights caused the mistakes
    optimizer.step()                     # Step E: nudge those weights to do better next time


Model size: 0.82 million parameters
step    0 | train loss 4.356 | val loss 4.349
step  300 | train loss 2.130 | val loss 2.159
step  600 | train loss 1.847 | val loss 1.939
step  900 | train loss 1.712 | val loss 1.859
step 1200 | train loss 1.636 | val loss 1.791
step 1500 | train loss 1.578 | val loss 1.734
step 1800 | train loss 1.530 | val loss 1.697
step 2100 | train loss 1.518 | val loss 1.686
step 2400 | train loss 1.466 | val loss 1.677
step 2700 | train loss 1.464 | val loss 1.671
step 2999 | train loss 1.442 | val loss 1.638


In [7]:
# ==========================================
# 7. LET IT WRITE SOME SHAKESPEARE
# ==========================================
model.eval()   # Exam mode - we're only using the model now, not training it

# Step A: Give it a starting prompt (you can change this!)
prompt = "To be or not to "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)   # Shape (1, T)

# Step B: Let it predict 500 more characters, one at a time
output_ids = model.generate(context, max_new_tokens=500)[0].tolist()

# Step C: Turn the numbers back into text
print(decode(output_ids))


To be or not to make that us
Againg this more: I speak triment blow from on Oxcition,
And they refrond. Turn, come theirses how, to thou,
The air touch's liegers how and weary;
I'll this head slaught me?

Provost:
Here coth medy to our live trie king,
In one woemphy ment? herefore have for to stains
her and yet consing tity of our proves.

DUKE VINCENTIO:
A pris of yet yours, Ongight men, eye my father's of patient,
And proforth of joys toy, thought nosign house and Capulent.

DUKE VINCENTIO:
Where idly the hea
